# RetailPulse -- Data Cleaning & Feature Engineering

**Objective:** Clean the raw Online Retail II dataset, engineer RFM features and rolling statistics, validate data quality, and export the processed dataset for downstream modeling.


In [1]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

FIGURES_DIR = os.path.join("..", "reports", "figures")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)


In [2]:
def save_fig(fig, name):
    path = os.path.join(FIGURES_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved: {name}")


## Load Raw Data

In [3]:
DATA_PATH = os.path.join("..", "data", "raw", "online_retail_II.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["InvoiceDate"])

print(f"Raw dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


Raw dataset: 525,461 rows x 8 columns


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Data Cleaning

Cleaning steps:
1. Remove cancelled/returned transactions (Invoice starting with 'C')
2. Drop rows with missing Customer ID (required for segmentation)
3. Remove zero or negative price rows
4. Remove zero or negative quantity rows
5. Drop duplicate rows
6. Handle missing descriptions


In [4]:
# Track cleaning progress
cleaning_log = []

initial_rows = len(df)
print(f"Starting rows: {initial_rows:,}")


Starting rows: 525,461


In [5]:
# Remove cancelled transactions (invoice starts with 'C')
cancelled_mask = df["Invoice"].astype(str).str.startswith("C")
cancelled_count = cancelled_mask.sum()
df = df[~cancelled_mask].copy()
cleaning_log.append(("Cancelled transactions removed", cancelled_count))
print(f"Removed {cancelled_count:,} cancelled transactions")
print(f"Remaining: {len(df):,} rows")


Removed 10,206 cancelled transactions
Remaining: 515,255 rows


In [6]:
# Remove rows without Customer ID
missing_cid = df["Customer ID"].isna().sum()
df = df.dropna(subset=["Customer ID"]).copy()
cleaning_log.append(("Missing Customer ID removed", missing_cid))
print(f"Removed {missing_cid:,} rows with missing Customer ID")
print(f"Remaining: {len(df):,} rows")


Removed 107,560 rows with missing Customer ID
Remaining: 407,695 rows


In [7]:
# Remove zero/negative prices
bad_price = (df["Price"] <= 0).sum()
df = df[df["Price"] > 0].copy()
cleaning_log.append(("Zero/negative price removed", bad_price))

# Remove zero/negative quantities
bad_qty = (df["Quantity"] <= 0).sum()
df = df[df["Quantity"] > 0].copy()
cleaning_log.append(("Zero/negative quantity removed", bad_qty))

print(f"Removed {bad_price:,} bad price rows and {bad_qty:,} bad quantity rows")
print(f"Remaining: {len(df):,} rows")


Removed 31 bad price rows and 0 bad quantity rows
Remaining: 407,664 rows


In [8]:
# Remove exact duplicates
dup_count = df.duplicated().sum()
df = df.drop_duplicates().copy()
cleaning_log.append(("Duplicate rows removed", dup_count))
print(f"Removed {dup_count:,} duplicate rows")
print(f"Remaining: {len(df):,} rows")


Removed 6,748 duplicate rows
Remaining: 400,916 rows


In [9]:
# Fill missing descriptions with 'Unknown'
missing_desc = df["Description"].isna().sum()
df["Description"] = df["Description"].fillna("Unknown")
cleaning_log.append(("Missing descriptions filled", missing_desc))
print(f"Filled {missing_desc:,} missing descriptions")


Filled 0 missing descriptions


In [10]:
# Cleaning summary
print("CLEANING SUMMARY")
print("-" * 55)
for step, count in cleaning_log:
    print(f"  {step:<40s} {count:>8,}")
print("-" * 55)
print(f"  {'Rows removed total':<40s} {initial_rows - len(df):>8,}")
print(f"  {'Final dataset size':<40s} {len(df):>8,}")
print(f"  Retention rate: {len(df)/initial_rows*100:.1f}%")


CLEANING SUMMARY
-------------------------------------------------------
  Cancelled transactions removed             10,206
  Missing Customer ID removed               107,560
  Zero/negative price removed                    31
  Zero/negative quantity removed                  0
  Duplicate rows removed                      6,748
  Missing descriptions filled                     0
-------------------------------------------------------
  Rows removed total                        124,545
  Final dataset size                        400,916
  Retention rate: 76.3%


## Feature Engineering

### Revenue and Date Features


In [11]:
# Revenue
df["Revenue"] = df["Quantity"] * df["Price"]

# Date-based features
df["Date"] = df["InvoiceDate"].dt.date
df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["DayOfWeek"] = df["InvoiceDate"].dt.dayofweek
df["Hour"] = df["InvoiceDate"].dt.hour
df["WeekOfYear"] = df["InvoiceDate"].dt.isocalendar().week.astype(int)

# Customer ID as integer
df["Customer ID"] = df["Customer ID"].astype(int)

print(f"Added Revenue and date features")
print(f"Columns: {list(df.columns)}")
df.head()


Added Revenue and date features
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Revenue', 'Date', 'Year', 'Month', 'DayOfWeek', 'Hour', 'WeekOfYear']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Date,Year,Month,DayOfWeek,Hour,WeekOfYear
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,2009-12-01,2009,12,1,7,49
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009-12-01,2009,12,1,7,49
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009-12-01,2009,12,1,7,49
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8,2009-12-01,2009,12,1,7,49
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,2009-12-01,2009,12,1,7,49


### RFM Feature Engineering

RFM (Recency, Frequency, Monetary) is a customer segmentation technique:
- **Recency**: How recently a customer made a purchase
- **Frequency**: How often they purchase
- **Monetary**: How much they spend


In [12]:
# Reference date: one day after the last transaction
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
print(f"Reference date for recency: {reference_date}")

rfm = df.groupby("Customer ID").agg(
    recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
    frequency=("Invoice", "nunique"),
    monetary=("Revenue", "sum")
).reset_index()

print(f"\nRFM table shape: {rfm.shape}")
rfm.describe().round(2)


Reference date for recency: 2010-12-10 20:01:00



RFM table shape: (4312, 4)


,Customer ID,recency,frequency,monetary
count,4312.00,4312.00,4312.00,4312.00
mean,15349.29,91.17,4.46,2040.41
std,1701.20,96.86,8.17,8911.76
min,12346.00,1.00,1.00,2.95
25%,13882.50,18.00,1.00,307.19
50%,15350.50,53.00,2.00,701.62
75%,16834.25,136.00,5.00,1714.93
max,18287.00,374.00,205.00,349164.35


In [13]:
# Assign RFM scores (1-5) using quantile-based binning
rfm["r_score"] = pd.qcut(rfm["recency"], q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["f_score"] = pd.qcut(rfm["frequency"].rank(method="first"), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary"], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Combined RFM score
rfm["rfm_score"] = rfm["r_score"] + rfm["f_score"] + rfm["m_score"]

# RFM segment string
rfm["rfm_segment"] = rfm["r_score"].astype(str) + rfm["f_score"].astype(str) + rfm["m_score"].astype(str)

print("RFM Score Distribution:")
print(rfm["rfm_score"].describe().round(2))
print()
rfm.head(10)


RFM Score Distribution:
count    4312.00
mean        9.02
std         3.58
min         3.00
25%         6.00
50%         9.00
75%        12.00
max        15.00
Name: rfm_score, dtype: float64



,Customer ID,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,rfm_segment
0,12346,165,11,372.86,2,5,2,9,252
1,12347,3,2,1323.32,5,2,4,11,524
2,12348,74,1,222.16,2,1,1,4,211
3,12349,43,3,2671.14,3,3,5,11,335
4,12351,11,1,300.93,5,1,2,8,512
5,12352,11,2,343.80,5,2,2,9,522
6,12353,44,1,317.76,3,1,2,6,312
7,12355,203,1,488.21,1,1,2,4,112
8,12356,16,3,3560.30,4,3,5,12,435
9,12357,24,2,12079.99,4,2,5,11,425


In [14]:
# Business-friendly segment labels based on RFM scores
def assign_segment(row):
    r, f, m = row["r_score"], row["f_score"], row["m_score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3 and m >= 3:
        return "Loyal Customers"
    elif r >= 4 and f <= 2:
        return "New Customers"
    elif r >= 3 and f >= 3 and m <= 2:
        return "Potential Loyalists"
    elif r <= 2 and f >= 3 and m >= 3:
        return "At Risk"
    elif r <= 2 and f <= 2 and m >= 3:
        return "Cant Lose Them"
    elif r <= 2 and f <= 2 and m <= 2:
        return "Lost"
    else:
        return "Others"

rfm["segment_label"] = rfm.apply(assign_segment, axis=1)

segment_counts = rfm["segment_label"].value_counts()
print("Customer Segments:")
print("-" * 40)
for seg, count in segment_counts.items():
    print(f"  {seg:<25s} {count:>5,} ({count/len(rfm)*100:.1f}%)")


Customer Segments:
----------------------------------------
  Champions                   926 (21.5%)
  Lost                        830 (19.2%)
  Loyal Customers             767 (17.8%)
  Others                      522 (12.1%)
  At Risk                     509 (11.8%)
  New Customers               367 (8.5%)
  Potential Loyalists         206 (4.8%)
  Cant Lose Them              185 (4.3%)


In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(rfm["recency"], bins=50, color="#3498db", edgecolor="white", alpha=0.8)
axes[0].set_title("Recency Distribution (days)")
axes[0].set_xlabel("Days since last purchase")
axes[0].set_ylabel("Customers")

axes[1].hist(rfm["frequency"].clip(upper=rfm["frequency"].quantile(0.95)),
             bins=40, color="#2ecc71", edgecolor="white", alpha=0.8)
axes[1].set_title("Frequency Distribution (clipped at 95th pct)")
axes[1].set_xlabel("Number of orders")
axes[1].set_ylabel("Customers")

axes[2].hist(rfm["monetary"].clip(upper=rfm["monetary"].quantile(0.95)),
             bins=40, color="#e74c3c", edgecolor="white", alpha=0.8)
axes[2].set_title("Monetary Distribution (clipped at 95th pct)")
axes[2].set_xlabel("Total revenue")
axes[2].set_ylabel("Customers")

fig.suptitle("RFM Feature Distributions", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "08_rfm_distributions.png")
plt.show()


Saved: 08_rfm_distributions.png


In [16]:
fig, ax = plt.subplots(figsize=(12, 6))

seg_order = segment_counts.sort_values(ascending=True)
colors = plt.cm.Set2(np.linspace(0, 1, len(seg_order)))
ax.barh(range(len(seg_order)), seg_order.values, color=colors)
ax.set_yticks(range(len(seg_order)))
ax.set_yticklabels(seg_order.index, fontsize=11)
ax.set_xlabel("Number of Customers")
ax.set_title("Customer Segments (RFM-based)", fontsize=16, fontweight="bold")

for i, v in enumerate(seg_order.values):
    ax.text(v + 10, i, f"{v:,}", va="center", fontsize=10)

fig.tight_layout()
save_fig(fig, "09_rfm_segments.png")
plt.show()


Saved: 09_rfm_segments.png


### Rolling Statistics

Compute daily aggregates and rolling window features for time-series modeling.


In [17]:
# Daily aggregated sales
daily_sales = df.groupby("Date").agg(
    total_revenue=("Revenue", "sum"),
    total_quantity=("Quantity", "sum"),
    transaction_count=("Invoice", "nunique"),
    unique_customers=("Customer ID", "nunique"),
    avg_order_value=("Revenue", "mean"),
).reset_index()

daily_sales = daily_sales.sort_values("Date").reset_index(drop=True)
print(f"Daily sales table: {daily_sales.shape[0]} days")
daily_sales.head()


Daily sales table: 307 days


,Date,total_revenue,total_quantity,transaction_count,unique_customers,avg_order_value
0,2009-12-01,43894.87,24335,98,91,20.284136
1,2009-12-02,52762.06,29679,110,94,23.491567
2,2009-12-03,67413.62,48009,122,106,28.698859
3,2009-12-04,33913.81,19954,80,76,17.943815
4,2009-12-05,9803.05,5119,30,26,24.507625


In [18]:
# Rolling window features
for window in [7, 14, 30]:
    daily_sales[f"revenue_ma_{window}d"] = (
        daily_sales["total_revenue"].rolling(window, min_periods=1).mean()
    )
    daily_sales[f"revenue_std_{window}d"] = (
        daily_sales["total_revenue"].rolling(window, min_periods=1).std()
    )
    daily_sales[f"qty_ma_{window}d"] = (
        daily_sales["total_quantity"].rolling(window, min_periods=1).mean()
    )

# Lag features
for lag in [1, 7, 14]:
    daily_sales[f"revenue_lag_{lag}d"] = daily_sales["total_revenue"].shift(lag)

# Day-of-week encoding
daily_sales["day_of_week"] = daily_sales["Date"].dt.dayofweek
daily_sales["is_weekend"] = daily_sales["day_of_week"].isin([5, 6]).astype(int)
daily_sales["month"] = daily_sales["Date"].dt.month
daily_sales["week_of_year"] = daily_sales["Date"].dt.isocalendar().week.astype(int)

print(f"Features added. Total columns: {daily_sales.shape[1]}")
print(f"Columns: {list(daily_sales.columns)}")


Features added. Total columns: 22
Columns: ['Date', 'total_revenue', 'total_quantity', 'transaction_count', 'unique_customers', 'avg_order_value', 'revenue_ma_7d', 'revenue_std_7d', 'qty_ma_7d', 'revenue_ma_14d', 'revenue_std_14d', 'qty_ma_14d', 'revenue_ma_30d', 'revenue_std_30d', 'qty_ma_30d', 'revenue_lag_1d', 'revenue_lag_7d', 'revenue_lag_14d', 'day_of_week', 'is_weekend', 'month', 'week_of_year']


In [19]:
fig, axes = plt.subplots(2, 1, figsize=(18, 10))

# Revenue with moving averages
axes[0].plot(daily_sales["Date"], daily_sales["total_revenue"],
             alpha=0.3, color="#bdc3c7", label="Daily")
axes[0].plot(daily_sales["Date"], daily_sales["revenue_ma_7d"],
             color="#3498db", linewidth=1.5, label="7-day MA")
axes[0].plot(daily_sales["Date"], daily_sales["revenue_ma_30d"],
             color="#e74c3c", linewidth=2, label="30-day MA")
axes[0].set_title("Daily Revenue with Moving Averages")
axes[0].set_ylabel("Revenue")
axes[0].legend()

# Rolling volatility
axes[1].plot(daily_sales["Date"], daily_sales["revenue_std_7d"],
             color="#9b59b6", linewidth=1.5, label="7-day Std")
axes[1].plot(daily_sales["Date"], daily_sales["revenue_std_30d"],
             color="#e67e22", linewidth=2, label="30-day Std")
axes[1].set_title("Revenue Volatility (Rolling Standard Deviation)")
axes[1].set_ylabel("Std Dev")
axes[1].set_xlabel("Date")
axes[1].legend()

fig.suptitle("Rolling Statistics -- Revenue Trends & Volatility",
             fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "10_rolling_statistics.png")
plt.show()


Saved: 10_rolling_statistics.png


## Data Validation

Run automated quality checks on the cleaned dataset to ensure it meets expectations for downstream modeling.


In [20]:
# Data validation checks
validation_results = []

def check(name, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    validation_results.append((name, status, detail))
    print(f"  [{status}] {name}" + (f" -- {detail}" if detail else ""))

print("DATA VALIDATION REPORT")
print("=" * 65)
print()

# Schema checks
print("Schema Checks:")
check("No missing Customer IDs",
      df["Customer ID"].isna().sum() == 0)
check("No missing InvoiceDate",
      df["InvoiceDate"].isna().sum() == 0)
check("Customer ID is integer",
      df["Customer ID"].dtype in [np.int64, np.int32])

print()

# Value range checks
print("Value Range Checks:")
check("All quantities positive",
      (df["Quantity"] > 0).all(),
      f"min={df['Quantity'].min()}")
check("All prices positive",
      (df["Price"] > 0).all(),
      f"min={df['Price'].min():.2f}")
check("All revenues positive",
      (df["Revenue"] > 0).all(),
      f"min={df['Revenue'].min():.2f}")
check("No cancelled invoices",
      ~df["Invoice"].astype(str).str.startswith("C").any())

print()

# Statistical checks
print("Statistical Checks:")
check("Sufficient customers (>1000)",
      df["Customer ID"].nunique() > 1000,
      f"count={df['Customer ID'].nunique():,}")
check("Sufficient products (>1000)",
      df["StockCode"].nunique() > 1000,
      f"count={df['StockCode'].nunique():,}")
check("Date range > 6 months",
      (df["InvoiceDate"].max() - df["InvoiceDate"].min()).days > 180,
      f"days={(df['InvoiceDate'].max() - df['InvoiceDate'].min()).days}")
check("No duplicate rows",
      df.duplicated().sum() == 0)

print()

# RFM checks
print("RFM Checks:")
check("RFM table has customers",
      len(rfm) > 0,
      f"count={len(rfm):,}")
check("RFM scores in valid range (3-15)",
      rfm["rfm_score"].between(3, 15).all(),
      f"range=[{rfm['rfm_score'].min()}, {rfm['rfm_score'].max()}]")
check("All customers have segment labels",
      rfm["segment_label"].notna().all())

print()

# Summary
passed = sum(1 for _, s, _ in validation_results if s == "PASS")
total = len(validation_results)
print(f"Results: {passed}/{total} checks passed")


DATA VALIDATION REPORT

Schema Checks:
  [PASS] No missing Customer IDs
  [PASS] No missing InvoiceDate
  [PASS] Customer ID is integer

Value Range Checks:
  [PASS] All quantities positive -- min=1
  [PASS] All prices positive -- min=0.00
  [PASS] All revenues positive -- min=0.00
  [PASS] No cancelled invoices

Statistical Checks:
  [PASS] Sufficient customers (>1000) -- count=4,312
  [PASS] Sufficient products (>1000) -- count=4,017
  [PASS] Date range > 6 months -- days=373


  [PASS] No duplicate rows

RFM Checks:
  [PASS] RFM table has customers -- count=4,312
  [PASS] RFM scores in valid range (3-15) -- range=[3, 15]
  [PASS] All customers have segment labels

Results: 14/14 checks passed


## Export Processed Data

In [21]:
# Save cleaned transaction data
clean_path = os.path.join(PROCESSED_DIR, "transactions_clean.csv")
df.to_csv(clean_path, index=False)
print(f"Saved cleaned transactions: {clean_path}")
print(f"  Shape: {df.shape}")

# Save RFM features
rfm_path = os.path.join(PROCESSED_DIR, "customer_rfm.csv")
rfm.to_csv(rfm_path, index=False)
print(f"\nSaved RFM features: {rfm_path}")
print(f"  Shape: {rfm.shape}")

# Save daily sales with rolling features
daily_path = os.path.join(PROCESSED_DIR, "daily_sales_features.csv")
daily_sales.to_csv(daily_path, index=False)
print(f"\nSaved daily sales features: {daily_path}")
print(f"  Shape: {daily_sales.shape}")


Saved cleaned transactions: ../data/processed/transactions_clean.csv
  Shape: (400916, 15)

Saved RFM features: ../data/processed/customer_rfm.csv
  Shape: (4312, 10)

Saved daily sales features: ../data/processed/daily_sales_features.csv
  Shape: (307, 22)


## Summary

In [22]:
print("DATA CLEANING & FEATURE ENGINEERING COMPLETE")
print("=" * 55)
print()
print(f"Raw dataset:        {initial_rows:>10,} rows")
print(f"Cleaned dataset:    {len(df):>10,} rows")
print(f"Retention rate:     {len(df)/initial_rows*100:>9.1f}%")
print()
print(f"Unique customers:   {df['Customer ID'].nunique():>10,}")
print(f"Unique products:    {df['StockCode'].nunique():>10,}")
print(f"Date range:         {df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()}")
print()
print("Exported files:")
print(f"  transactions_clean.csv     ({df.shape[0]:,} rows x {df.shape[1]} cols)")
print(f"  customer_rfm.csv           ({rfm.shape[0]:,} rows x {rfm.shape[1]} cols)")
print(f"  daily_sales_features.csv   ({daily_sales.shape[0]:,} rows x {daily_sales.shape[1]} cols)")
print()
print("Next steps:")
print("  - Customer segmentation using K-Means and DBSCAN on RFM features")
print("  - Time-series preparation for demand forecasting")


DATA CLEANING & FEATURE ENGINEERING COMPLETE

Raw dataset:           525,461 rows
Cleaned dataset:       400,916 rows
Retention rate:          76.3%

Unique customers:        4,312
Unique products:         4,017
Date range:         2009-12-01 to 2010-12-09

Exported files:
  transactions_clean.csv     (400,916 rows x 15 cols)
  customer_rfm.csv           (4,312 rows x 10 cols)
  daily_sales_features.csv   (307 rows x 22 cols)

Next steps:
  - Customer segmentation using K-Means and DBSCAN on RFM features
  - Time-series preparation for demand forecasting
